# EDA 7.001: Categorical counts over first 30M rows

Goal: scan the raw Steam reviews CSV in **1M-row chunks**, up to a cap of
**30M rows**, and accumulate value counts for categorical-ish columns.

This is a lightweight, streaming-style view of the global shape of the
dataset without trying to load everything into memory at once.

## Imports and configuration

In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, List

import pandas as pd

from steam_review_ml.data.loaders import load_raw_reviews

RAW_PATH = Path("../../data/raw/steam_reviews_full.csv")

# Streaming / counting config
CHUNK_ROWS = 1_000_000
MAX_ROWS = 25_000_000  # stop after this many rows (or EOF, whichever comes first)
TOP_K = 20  # how many most-common values to show per column

print("Raw path:", RAW_PATH.resolve())

pd.set_option("display.max_columns", None)

Raw path: /home/ryanr/workspace/steam_recommendations/data/raw/steam_reviews_full.csv


In [5]:
! wc -l ../../data/raw/steam_reviews_full.csv

48296896 ../../data/raw/steam_reviews_full.csv


In [24]:
CAT_COLS = [
    "app_name",
    "review_id",
    "language",
    "votes_helpful",
    "votes_funny",
    "author.num_games_owned",
    "author.num_reviews",
]

## Streaming pass: count categorical values in chunks

We use `pd.read_csv(..., chunksize=CHUNK_ROWS)` directly instead of
`load_raw_reviews` so that we can incrementally read and process 1M-row
+rows at a time.

In [25]:
if not RAW_PATH.exists():
    raise FileNotFoundError(f"Raw reviews file not found at {RAW_PATH}")

# Counter per column
counters: Dict[str, Counter] = defaultdict(Counter)

rows_seen = 0
chunk_idx = 0

for chunk in pd.read_csv(RAW_PATH, chunksize=CHUNK_ROWS):
    chunk_idx += 1
    rows_seen += len(chunk)
    print(
        f"Chunk {chunk_idx}: rows {rows_seen - len(chunk)}–{rows_seen - 1} (size={len(chunk)})"
    )

    for col in CAT_COLS:
        # Drop NaNs and cast to string for consistent Counter keys
        series = chunk[col].dropna().astype(str)
        counters[col].update(series.values)

    if rows_seen >= MAX_ROWS:
        print(f"Reached MAX_ROWS={MAX_ROWS}, stopping.")
        break

print(f"Total rows processed: {rows_seen}")

Chunk 1: rows 0–999999 (size=1000000)
Chunk 2: rows 1000000–1999999 (size=1000000)
Chunk 3: rows 2000000–2999999 (size=1000000)
Chunk 4: rows 3000000–3999999 (size=1000000)
Chunk 5: rows 4000000–4999999 (size=1000000)
Chunk 6: rows 5000000–5999999 (size=1000000)
Chunk 7: rows 6000000–6999999 (size=1000000)
Chunk 8: rows 7000000–7999999 (size=1000000)
Chunk 9: rows 8000000–8999999 (size=1000000)
Chunk 10: rows 9000000–9999999 (size=1000000)
Chunk 11: rows 10000000–10999999 (size=1000000)
Chunk 12: rows 11000000–11999999 (size=1000000)
Chunk 13: rows 12000000–12999999 (size=1000000)
Chunk 14: rows 13000000–13999999 (size=1000000)
Chunk 15: rows 14000000–14999999 (size=1000000)
Chunk 16: rows 15000000–15999999 (size=1000000)
Chunk 17: rows 16000000–16999999 (size=1000000)
Chunk 18: rows 17000000–17999999 (size=1000000)
Chunk 19: rows 18000000–18999999 (size=1000000)
Chunk 20: rows 19000000–19999999 (size=1000000)
Chunk 21: rows 20000000–20999999 (size=1000000)
Chunk 22: rows 21000000–2174

### Summary: top values per categorical column

Display the `TOP_K` most-common values for each counted column.

In [33]:
# Get the count of review_id's from the counters dictionary
review_id_count = sum(counters["review_id"].values())
review_id_num_keys = len(counters["review_id"])
print(f"Total number of review_id's counted: {review_id_count}")
print(f"Number of unique review_id's: {review_id_num_keys}")

Total number of review_id's counted: 21747371
Number of unique review_id's: 21612444


In [26]:
for col, counter in counters.items():
    print("\n===", col, "===")
    most_common = counter.most_common(TOP_K)
    df_counts = pd.DataFrame(most_common, columns=[col, "count"])
    display(df_counts)


=== app_name ===


,app_name,count
0,PLAYERUNKNOWN'S BATTLEGROUNDS,1644255
1,Grand Theft Auto V,1019116
2,Tom Clancy's Rainbow Six Siege,841918
3,Terraria,672815
4,Garry's Mod,655524
5,Rust,549074
6,Rocket League,498565
7,PAYDAY 2,487747
8,Among Us,485293
9,The Witcher 3: Wild Hunt,469395



=== review_id ===


,review_id,count
0,84774500,2
1,84774347,2
2,84774076,2
3,84773959,2
4,84773690,2
5,84773587,2
6,84773385,2
7,84773322,2
8,84773301,2
9,84773234,2



=== language ===


...

,votes_funny,count
0,0,19144299
1,1,1719903
2,2,381143
3,3,144356
4,4,74835
5,5,44516
6,6,30585
7,7,22478
8,8,17680
9,9,13681



=== author.num_games_owned ===


,author.num_games_owned,count
0,1,385963
1,2,332333
2,3,314674
3,4,300465
4,5,287751
5,6,278254
6,7,271095
7,8,263581
8,9,256059
9,10,251651



=== author.num_reviews ===


,author.num_reviews,count
0,1,4926031
1,2,3062596
2,3,2193765
3,4,1658093
4,5,1290267
5,6,1029443
6,7,831049
7,8,685358
8,9,571842
9,10,490896


# Raw File Review

In [13]:
lastrow = 21_747_371

In [14]:
chunk = pd.read_csv(RAW_PATH, skiprows=lastrow - 3, nrows=10)

In [15]:
display(chunk)

,21747372,546560,Half-Life: Alyx,65645115,english,"A tripod thing bears down on you, it looks immense and the sound design ( on Index speakers ) is epic. A combine guard tells you to keep yours hands up and tracks you with his head, put your hands down and he barks at you to keep your hands up. Welcome to the movie . . . Nothing in VR gaming has ever felt anything like this. The hype may be true",1584986693,1584986693.1,True,0,...,False,False.1,False.2,76561198046495482,70,10,3296.0,15.0,24.0,1611080231.0
0,21747373,546560,Half-Life: Alyx,65645100,english,Honestly this is the best vr game ever,1584986673,1584986673,True,0,...,True,False,False,76561198176379749,79,17,2654.0,0.0,34.0,1.591634e+09
1,21747374,546560,Half-Life: Alyx,65645066,english,Smooth turning is not working right now.\nIt a...,1584986631,1586382422,True,0,...,True,False,False,76561198041763187,140,3,210.0,0.0,12.0,1.589715e+09
2,21747375,546560,Half-Life: Alyx,65644930,schinese,WMR加载中闪退，无法进入游戏\nWMR Flashback in Load，cant pl...,1584986505,1592060800,True,0,...,True,False,False,76561198116747069,253,7,1062.0,0.0,8.0,1.602858e+09


In [19]:
# Show 20 lines centered roughly around 21,747,371
! sed -n '21747350,21747390p' ../../data/raw/steam_reviews_full.csv

10371901,588650,Dead Cells,33065091,english,"Holy iSH, this game is FANTASTIC!!!!  Many fun!  Such Challenging! WoW!",1498876606,1498876606,True,1,1,0.5,0,True,False,True,76561197977341551,364,35,351.0,0.0,289.0,1590180825.0
10371902,588650,Dead Cells,33064758,english,"Fun game to start. Challenging but still a great way to start. I cant wait to get more updates
",1498875763,1498875763,True,0,0,0.0,0,True,False,True,76561198097525966,69,3,87.0,0.0,58.0,1538418839.0
10371903,588650,Dead Cells,33064669,schinese,这游戏……有、有毒！！,1498875558,1498875558,True,0,0,0.0,0,True,False,True,76561198001610528,422,14,6084.0,0.0,479.0,1601537471.0
10371904,588650,Dead Cells,33064434,english,"I've been enjoying dead cells a lot! Beautiful, challenging, and every run feels different.

Though I will admit--the fact that you get infinite grenades does ruin it for me a little bit. The fact that a fresh character can--with no fancy weapons or upgrades or l33t dodging skills--take care of the hardest levels in th

In [20]:
! awk 'NR>=21747350 && NR<=21747390 { printf "%d:%s\n", NR, $0 }' \
  ../../data/raw/steam_reviews_full.csv

21747350:10371901,588650,Dead Cells,33065091,english,"Holy iSH, this game is FANTASTIC!!!!  Many fun!  Such Challenging! WoW!",1498876606,1498876606,True,1,1,0.5,0,True,False,True,76561197977341551,364,35,351.0,0.0,289.0,1590180825.0
21747351:10371902,588650,Dead Cells,33064758,english,"Fun game to start. Challenging but still a great way to start. I cant wait to get more updates
21747352:",1498875763,1498875763,True,0,0,0.0,0,True,False,True,76561198097525966,69,3,87.0,0.0,58.0,1538418839.0
21747353:10371903,588650,Dead Cells,33064669,schinese,这游戏……有、有毒！！,1498875558,1498875558,True,0,0,0.0,0,True,False,True,76561198001610528,422,14,6084.0,0.0,479.0,1601537471.0
21747354:10371904,588650,Dead Cells,33064434,english,"I've been enjoying dead cells a lot! Beautiful, challenging, and every run feels different.
21747355:
21747356:Though I will admit--the fact that you get infinite grenades does ruin it for me a little bit. The fact that a fresh character can--with no fancy weapons or upgrade

In [21]:
# For each unique app_name, find the min and max index (row number) in the full dataset.
# We'll use pd.read_csv with chunks since the file is large.
from collections import defaultdict

app_name_top_idx = defaultdict(lambda: None)
app_name_bottom_idx = defaultdict(lambda: None)

chunksize = 1_000_000
row_offset = 0

for chunk in pd.read_csv(RAW_PATH, usecols=["app_name"], chunksize=chunksize):
    for idx, app in enumerate(chunk["app_name"]):
        abs_idx = row_offset + idx
        # Top: If not set, this is the earliest row for this app
        if app_name_top_idx[app] is None:
            app_name_top_idx[app] = abs_idx
        # Bottom: Always update, so ends up as latest row for this app
        app_name_bottom_idx[app] = abs_idx
    row_offset += len(chunk)

# Now show as a DataFrame
app_entry_bounds = pd.DataFrame(
    {
        "app_name": list(app_name_top_idx.keys()),
        "top_entry": [app_name_top_idx[a] for a in app_name_top_idx.keys()],
        "bottom_entry": [app_name_bottom_idx[a] for a in app_name_top_idx.keys()],
    }
)

app_entry_bounds = app_entry_bounds.sort_values("top_entry").reset_index(drop=True)
display(app_entry_bounds.head(20))  # Show first 20 as an example

,app_name,top_entry,bottom_entry
0,The Witcher 3: Wild Hunt,0,469394
1,Half-Life,469395,526714
2,Counter-Strike: Source,526715,644795
3,Half-Life 2: Episode Two,644796,668295
4,Portal 2,668296,900624
5,X Rebirth,900625,907625
6,Garry's Mod,907626,1563149
7,Sid Meier's Civilization V,1563150,1734553
8,Dead by Daylight,1734554,2153450
9,Sid Meier's Civilization VI,2153451,2297239


In [23]:
# first 10 rows of the dataset
! head -n 10 ../../data/raw/steam_reviews_full.csv
# last 10 rows of the dataset
! tail -n 10 ../../data/raw/steam_reviews_full.csv

,app_id,app_name,review_id,language,review,timestamp_created,timestamp_updated,recommended,votes_helpful,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,author.steamid,author.num_games_owned,author.num_reviews,author.playtime_forever,author.playtime_last_two_weeks,author.playtime_at_review,author.last_played
0,292030,The Witcher 3: Wild Hunt,85185598,schinese,不玩此生遗憾，RPG游戏里的天花板，太吸引人了,1611381629,1611381629,True,0,0,0.0,0,True,False,False,76561199095369542,6,2,1909.0,1448.0,1909.0,1611343383.0
1,292030,The Witcher 3: Wild Hunt,85185250,schinese,拔DIAO无情打桩机--杰洛特!!!,1611381030,1611381030,True,0,0,0.0,0,True,False,False,76561198949504115,30,10,2764.0,2743.0,2674.0,1611386307.0
2,292030,The Witcher 3: Wild Hunt,85185111,schinese,巫师3NB,1611380800,1611380800,True,0,0,0.0,0,True,False,False,76561199090098988,5,1,1061.0,1061.0,1060.0,1611383777.0
3,292030,The Witcher 3: Wild Hunt,85184605,english,"One of the best RPG's of all time, worthy 

# Duplicate review_id's

In [42]:
# Pass 1: count review_id occurrences (streaming)

from collections import Counter

review_id_counts = Counter()
rows_seen = 0
chunksize = 1_000_000

for chunk in pd.read_csv(RAW_PATH, usecols=["review_id"], chunksize=chunksize):
    ids = chunk["review_id"].astype(str)
    review_id_counts.update(ids.values)
    rows_seen += len(chunk)
    if rows_seen % 5_000_000 == 0:
        print(f"Counted review_id's for {rows_seen} rows so far...")

# Keep only IDs that appear more than once
duplicate_ids = {rid for rid, c in review_id_counts.items() if c > 1}
print(f"Total unique review_id values: {len(review_id_counts)}")
print(f"Unique duplicate review_id values (count>1): {len(duplicate_ids)}")

# Pass 2: collect all rows whose review_id is in duplicate_ids

chunksize = 1_000_000
rows_seen = 0

dupe_rows = []

for chunk in pd.read_csv(RAW_PATH, chunksize=chunksize):
    mask = chunk["review_id"].astype(str).isin(duplicate_ids)
    dupes = chunk[mask]
    if not dupes.empty:
        dupe_rows.append(dupes)
    rows_seen += len(chunk)
    if rows_seen % 5_000_000 == 0:
        print(f"Scanned {rows_seen} rows for duplicate review_id records...")

if dupe_rows:
    dupe_review_df = pd.concat(dupe_rows, ignore_index=True)
    # sort dupe_review_df by review_id
    dupe_review_df = dupe_review_df.sort_values("review_id")
    # drop 'Unnamed: 0' column
    dupe_review_df = dupe_review_df.drop(columns=["Unnamed: 0"])
    print(f"Total rows with duplicate review_id: {len(dupe_review_df)}")
else:
    print("No duplicate review_id's found.")

Counted review_id's for 5000000 rows so far...
Counted review_id's for 10000000 rows so far...
Counted review_id's for 15000000 rows so far...
Counted review_id's for 20000000 rows so far...
Total unique review_id values: 21612444
Unique duplicate review_id values (count>1): 134927
Scanned 5000000 rows for duplicate review_id records...
Scanned 10000000 rows for duplicate review_id records...
Scanned 15000000 rows for duplicate review_id records...
Scanned 20000000 rows for duplicate review_id records...
Total rows with duplicate review_id: 269854


In [43]:
(
    dupe_review_df.shape[0],
    dupe_review_df.drop_duplicates(subset=["review_id"]).shape[0],
    dupe_review_df.drop_duplicates(subset=["review_id"]).shape[0] * 2,
)

(269854, 134927, 269854)

In [46]:
# Check if the duplicate records for each review_id are actually identical.
# For each review_id with duplicates, compare the rows and find differing columns.

# First, find all review_ids with more than one occurrence
dupe_counts = dupe_review_df["review_id"].value_counts()
multi_dupe_ids = dupe_counts[dupe_counts > 1].index

# Pick a sample of duplicate review_id groups to inspect (for demonstration, first 5 IDs)
sample_ids = list(multi_dupe_ids[:5])

for rid in sample_ids:
    group = dupe_review_df[dupe_review_df["review_id"] == rid]
    # Compare all rows in the group
    if group.nunique(dropna=False).max() == 1:
        print(f"review_id {rid}: all rows are identical")
    else:
        # Identify fields that differ
        diff_columns = group.nunique(dropna=False) > 1
        differing_fields = group.columns[diff_columns]
        print(f"review_id {rid} has differences in columns: {list(differing_fields)}")
        display(group)  # Show the differing rows for examination

# If you want to check all duplicate groups (may be slow!), you can collect stats as below:
diff_stats = {}
for rid, group in dupe_review_df.groupby("review_id"):
    nunique = group.nunique(dropna=False)
    diff_cols = nunique[nunique > 1].index.tolist()
    if diff_cols:
        diff_stats[rid] = diff_cols

print(
    f"\nAmong {len(multi_dupe_ids)} duplicate review_id's, {len(diff_stats)} have non-identical rows."
)
if diff_stats:
    print("Example differing columns (sample):")
    for rid, cols in list(diff_stats.items())[:5]:
        print(f"  review_id {rid}: differs in {cols}")

review_id 30150973: all rows are identical
review_id 30151071: all rows are identical
review_id 30151511: all rows are identical
review_id 30151527: all rows are identical
review_id 30151623: all rows are identical

Among 134927 duplicate review_id's, 0 have non-identical rows.


In [1]:
134927 / 21747371

0.006204290164544487

In [47]:
display(dupe_review_df.head(10))

,app_id,app_name,review_id,language,review,timestamp_created,timestamp_updated,recommended,votes_helpful,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,author.steamid,author.num_games_owned,author.num_reviews,author.playtime_forever,author.playtime_last_two_weeks,author.playtime_at_review,author.last_played
133397,367520,Hollow Knight,30150973,english,A great 2D adventure game. I've played less th...,1487971912,1487971912,True,0,0,0.499931,0,True,False,False,76561198012123927,746,47,2404.0,0.0,119.0,1.551587e+09
268324,367520,Hollow Knight,30150973,english,A great 2D adventure game. I've played less th...,1487971912,1487971912,True,0,0,0.499931,0,True,False,False,76561198012123927,746,47,2404.0,0.0,119.0,1.551587e+09
268323,367520,Hollow Knight,30151071,english,Alright everyone this is going to be a long re...,1487972199,1487972199,True,0,0,0.499931,0,False,False,False,76561198138269311,99,1,528.0,0.0,71.0,1.543720e+09
133396,367520,Hollow Knight,30151071,english,Alright everyone this is going to be a long re...,1487972199,1487972199,True,0,0,0.499931,0,False,False,False,76561198138269311,99,1,528.0,0.0,71.0,1.543720e+09
133395,367520,Hollow Knight,30151511,english,I love it so far!,1487973492,1487973492,True,0,0,0.474138,0,True,False,False,76561198359352806,67,3,2488.0,0.0,32.0,1.514554e+09
268322,367520,Hollow Knight,30151511,english,I love it so far!,1487973492,1487973492,True,0,0,0.474138,0,True,False,False,76561198359352806,67,3,2488.0,0.0,32.0,1.514554e+09
268321,367520,Hollow Knight,30151527,english,When faced by an angry bug the only possible s...,1487973524,1487973524,True,0,0,0.499931,0,True,False,False,76561198016861078,392,15,1075.0,0.0,382.0,1.517434e+09
133394,367520,Hollow Knight,30151527,english,When faced by an angry bug the only possible s...,1487973524,1487973524,True,0,0,0.499931,0,True,False,False,76561198016861078,392,15,1075.0,0.0,382.0,1.517434e+09
133393,367520,Hollow Knight,30151623,russian,"Годный платформер, поиграл пару часов и прям а...",1487973844,1487973894,True,4,0,0.482792,0,True,False,False,76561198045494132,220,15,1913.0,0.0,1323.0,1.570829e+09
268320,367520,Hollow Knight,30151623,russian,"Годный платформер, поиграл пару часов и прям а...",1487973844,1487973894,True,4,0,0.482792,0,True,False,False,76561198045494132,220,15,1913.0,0.0,1323.0,1.570829e+09
